# 01 — Algebra Setup

**Hypothesis:** In Cl(7,5,1) with white keys positive and black keys negative, the C-major triad has positive squared norm, chords with an odd number of black keys have negative squared norm, and the full chromatic cluster has norm² = −1 (5 black keys, odd count).

In [1]:
from kingdon import Algebra
import numpy as np

alg = Algebra(7, 5, 1)

# kingdon Algebra(7,5,1): signature = [0,+1,...,+1,-1,...,-1]
# e0=null(OCTAVE), e1-e7=positive(white keys), e8,e9,eA,eB,eC=negative(black keys)
def mv(k): return alg.multivector({k: 1})

PITCH = {
    'C':  mv('e1'), 'D':  mv('e2'), 'E':  mv('e3'), 'F':  mv('e4'),
    'G':  mv('e5'), 'A':  mv('e6'), 'B':  mv('e7'),
    'Cs': mv('e8'), 'Ds': mv('e9'), 'Fs': mv('eA'), 'Gs': mv('eB'), 'As': mv('eC'),
}
OCTAVE = mv('e0')

def norm2(x): return (x * ~x).e
def is_zero(x, tol=1e-10): return all(abs(float(v)) <= tol for v in x.values())
def blade_coeffs(x, tol=1e-10): return {alg.bin2canon[k]: float(v) for k, v in x.items() if abs(float(v)) > tol}

print('Algebra Cl(7,5,1) ready. Signature:', alg.signature)

Algebra Cl(7,5,1) ready. Signature: [0, 1, 1, 1, 1, 1, 1, 1, -1, -1, -1, -1, -1]


In [2]:
# Standard triads (grade-3 blades)
C_major  = PITCH['C'] ^ PITCH['E'] ^ PITCH['G']
C_minor  = PITCH['C'] ^ PITCH['Ds'] ^ PITCH['G']
C_dim    = PITCH['C'] ^ PITCH['Ds'] ^ PITCH['Fs']
C_aug    = PITCH['C'] ^ PITCH['E'] ^ PITCH['Gs']

# Seventh chords (grade-4 blades)
C_maj7   = PITCH['C'] ^ PITCH['E'] ^ PITCH['G'] ^ PITCH['B']
C_dom7   = PITCH['C'] ^ PITCH['E'] ^ PITCH['G'] ^ PITCH['As']
C_min7   = PITCH['C'] ^ PITCH['Ds'] ^ PITCH['G'] ^ PITCH['As']
C_dim7   = PITCH['C'] ^ PITCH['Ds'] ^ PITCH['Fs'] ^ PITCH['A']
C_hdim7  = PITCH['C'] ^ PITCH['Ds'] ^ PITCH['Fs'] ^ PITCH['As']

# Full chromatic cluster (grade-12 blade)
chromatic = (PITCH['C'] ^ PITCH['Cs'] ^ PITCH['D'] ^ PITCH['Ds'] ^ PITCH['E'] ^ PITCH['F'] ^
             PITCH['Fs'] ^ PITCH['G'] ^ PITCH['Gs'] ^ PITCH['A'] ^ PITCH['As'] ^ PITCH['B'])

print('Chords constructed.')

Chords constructed.


In [3]:
# Claim 1: C-major triad (all white keys) has positive squared norm
n2_major = norm2(C_major)
print(f'‖C_major‖² = {n2_major}  (C E G — 0 black keys, (−1)^0 = +1)')
print('✓' if n2_major > 0 else '✗')

‖C_major‖² = 1  (C E G — 0 black keys, (−1)^0 = +1)
✓


In [4]:
# Claim 2: norm² = (−1)^(number of black-key tones)
# C_minor has 1 black key (Ds) → norm² = −1
# C_dim has 2 black keys (Ds, Fs) → norm² = +1
n2_minor = norm2(C_minor)
n2_dim   = norm2(C_dim)
n2_aug   = norm2(C_aug)

print(f'‖C_minor‖² = {n2_minor}  (1 black key Ds)')
print(f'‖C_dim‖²   = {n2_dim}  (2 black keys Ds, Fs)')
print(f'‖C_aug‖²   = {n2_aug}  (1 black key Gs)')
print()
ok = n2_minor == -1 and n2_dim == 1 and n2_aug == -1
print('✓ parity rule holds' if ok else '✗')

‖C_minor‖² = -1  (1 black key Ds)
‖C_dim‖²   = 1  (2 black keys Ds, Fs)
‖C_aug‖²   = -1  (1 black key Gs)

✓ parity rule holds


In [5]:
# Claim 3: chromatic cluster has norm² = −1 (5 black keys, odd count)
n2_chrom = norm2(chromatic)
print(f'‖chromatic‖² = {n2_chrom}  (5 black keys, (−1)^5 = −1)')
print('✓' if n2_chrom == -1 else '✗')

‖chromatic‖² = -1  (5 black keys, (−1)^5 = −1)
✓


In [6]:
# Survey all chords
chords = {
    'C_major': C_major, 'C_minor': C_minor, 'C_dim': C_dim, 'C_aug': C_aug,
    'C_maj7':  C_maj7,  'C_dom7': C_dom7,  'C_min7': C_min7,
    'C_dim7':  C_dim7,  'C_hdim7': C_hdim7,
}
print(f'{"Chord":<12} {"black keys":>12} {"‖·‖²":>6}')
print('-' * 32)
black = {'Cs', 'Ds', 'Fs', 'Gs', 'As'}
tones_map = {
    'C_major': ('C','E','G'),       'C_minor': ('C','Ds','G'),
    'C_dim':   ('C','Ds','Fs'),     'C_aug':   ('C','E','Gs'),
    'C_maj7':  ('C','E','G','B'),   'C_dom7':  ('C','E','G','As'),
    'C_min7':  ('C','Ds','G','As'), 'C_dim7':  ('C','Ds','Fs','A'),
    'C_hdim7': ('C','Ds','Fs','As'),
}
for name, ch in chords.items():
    nblack = sum(1 for t in tones_map[name] if t in black)
    n2 = norm2(ch)
    expected = (-1)**nblack
    marker = '✓' if n2 == expected else '✗'
    print(f'{name:<12} {nblack:>12} {n2:>6}  {marker}')

Chord          black keys   ‖·‖²
--------------------------------
C_major                 0      1  ✓
C_minor                 1     -1  ✓
C_dim                   2      1  ✓
C_aug                   1     -1  ✓
C_maj7                  0      1  ✓
C_dom7                  1     -1  ✓
C_min7                  2      1  ✓
C_dim7                  2      1  ✓
C_hdim7                 3     -1  ✓


## Discussion

The squared norm of a pure blade is the product of the squares of its constituent basis vectors:
$$\|B\|^2 = \prod_i e_i^2 = (-1)^{\text{\# black-key tones}}$$

- **White-only chords** (C-major, C-maj7): norm² = +1 — fully within the positive subspace.
- **Odd black-key count** (C-minor, C-aug, C-dom7, C-hdim7): norm² = −1 — the metric is Lorentzian in those directions.
- **Even black-key count** (C-dim, C-min7, C-dim7): norm² = +1 — the two negative signatures cancel.
- **Chromatic cluster** (5 black keys): norm² = −1.

**Held.** The squared norm is a clean parity invariant of the chord's black-key content. This is the algebraic fingerprint of C-major as the reference key: it is the unique major triad with norm² = +1.